In [ ]:
# ============================================================
# CHATBOT BANCOESTADO - PRUEBA 2 ING.SOLUCIONES CON IA
# Integración de mejoras de Prompt Engineering:
# 1) GitHub Models API (usado como backend del modelo)
# 2) LangChain Model API (gestión del modelo y mensajes)
# 3) LangChain Streaming (respuesta en tiempo real)
# 4) LangChain Memory (memoria conversacional)
# 5) Prompt Engineering (Zero-shot + Few-shot)
# ============================================================

# ----------------------------
# IMPORTS
# ----------------------------
import os
import time

# Cliente OpenAI (usado indirectamente a través de LangChain con GitHub Models)
from openai import OpenAI

# LangChain para trabajar con modelos de lenguaje
from langchain_openai import ChatOpenAI

# Tipos de mensajes (roles: system, user, assistant)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Memoria conversacional
from langchain_core.chat_history import InMemoryChatMessageHistory

# ----------------------------
# 1. CONFIGURACIÓN GITHUB MODELS API
# ----------------------------
print("=== 1. Configuración GitHub Models API ===")

# Variables de entorno (seguridad: no hardcodear credenciales)
github_base_url = os.getenv("GITHUB_BASE_URL") or os.getenv("OPENAI_BASE_URL")
github_token = os.getenv("GITHUB_TOKEN")

# Validación de variables
if not github_base_url:
    raise ValueError("No se encontró GITHUB_BASE_URL ni OPENAI_BASE_URL.")

if not github_token:
    raise ValueError("No se encontró GITHUB_TOKEN.")

# Cliente OpenAI (opcional, pero útil para pruebas directas)
client = OpenAI(
    base_url=github_base_url,
    api_key=github_token
)

print("Base URL:", github_base_url)
print("API Key configurada:", "✓" if github_token else "✗")

# ----------------------------
# 2. CONFIGURACIÓN LANGCHAIN MODEL API
# ----------------------------
print("\n=== 2. Configuración LangChain Model API ===")

# Modelo LLM usando LangChain (usa GitHub Models como backend)
llm = ChatOpenAI(
    base_url=github_base_url,
    api_key=github_token,
    model="gpt-4o",
    temperature=0.1,   # baja creatividad → mayor precisión
    max_tokens=300     # evita respuestas cortadas
)

print("Modelo LangChain configurado:", llm.model_name)

# ----------------------------
# 3. CONFIGURACIÓN DE MEMORIA
# ----------------------------
print("\n=== 3. Configuración de memoria conversacional ===")

# Almacén de sesiones
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# ----------------------------
# 4. PROMPT ENGINEERING
# ----------------------------

# 🔹 Prompt principal (ZERO-SHOT)
SYSTEM_PROMPT = """
Eres un asistente virtual de BancoEstado.

TAREA:
Responder preguntas sobre:
- cuentas bancarias
- tarjetas
- transferencias
- servicios digitales

OBJETIVO:
Respuestas claras, breves, seguras y fáciles de entender.

FORMATO:
1. Respuesta principal
2. Pasos (si aplica)
3. Advertencia de seguridad
4. Canal oficial

RESTRICCIONES:
- No pedir datos sensibles
- No inventar información
- Derivar a canales oficiales si es necesario

REGLA IMPORTANTE:
Puedes utilizar el historial de la conversación para responder preguntas sobre mensajes anteriores del usuario.
"""

# 🔹 Ejemplos (FEW-SHOT) 
FEW_SHOT_MESSAGES = [

    HumanMessage(content="¿Qué hago si pierdo mi tarjeta?"), 
    AIMessage(content="""1. Respuesta principal:
Debes bloquear tu tarjeta inmediatamente.

2. Pasos:
- Ingresa a la app o web de BancoEstado
- Contacta atención al cliente
- Solicita reposición

3. Advertencia:
No compartas datos personales.

4. Canal oficial:
App BancoEstado o sitio web oficial."""),

    HumanMessage(content="¿Cómo creo una CuentaRUT?"),
    AIMessage(content="""1. Respuesta principal:
Puedes solicitar la CuentaRUT si cumples los requisitos.

2. Pasos:
- Revisa requisitos en el sitio oficial
- Solicita online o presencial
- Activa tu cuenta

3. Advertencia:
Usa solo canales oficiales.

4. Canal oficial:
Sitio web o sucursal BancoEstado.""")
]

# ----------------------------
# 5. STREAMING + MEMORIA
# ----------------------------
print("\n=== 4. Streaming habilitado ===")

def responder_con_streaming_y_memoria(pregunta: str, session_id: str):

    # Obtener historial de conversación
    history = get_session_history(session_id)

    # Construcción de mensajes
    mensajes = (
        [SystemMessage(content=SYSTEM_PROMPT)]
        + FEW_SHOT_MESSAGES
        + history.messages
        + [HumanMessage(content=pregunta)]
    )

    respuesta_texto = ""

    # Streaming token por token
    for chunk in llm.stream(mensajes):
        contenido = chunk.content
        if contenido:
            print(contenido, end="", flush=True)
            respuesta_texto += contenido
            time.sleep(0.1)

    # Guardar en memoria
    history.add_user_message(pregunta)
    history.add_ai_message(respuesta_texto)

    return respuesta_texto

# ----------------------------
# 6. CHATBOT INTERACTIVO
# ----------------------------
print("\n=== CHATBOT INTERACTIVO BANCOESTADO ===")
print("Escribe tu consulta y presiona Enter.")
print("Para terminar, escribe: salir\n")

session_id = "bancoestado_chat"

while True:
    pregunta = input("Tú: ").strip()

    if pregunta.lower() == "salir":
        print("Chatbot: Hasta luego. Gracias por usar el asistente virtual de BancoEstado.")
        break

    if not pregunta:
        print("Chatbot: Por favor, escribe una consulta válida.")
        continue

    print("Chatbot: ", end="", flush=True)

    try:
        responder_con_streaming_y_memoria(pregunta, session_id)
        print("\n")

    except Exception as e:
        print(f"\nError: {e}")
        print("Verifica tu configuración o credenciales.\n")

# ----------------------------
# 7. HISTORIAL FINAL
# ----------------------------
print("\n=== HISTORIAL DE LA CONVERSACIÓN ===")

historial = store.get(session_id, InMemoryChatMessageHistory()).messages

if historial:
    for i, msg in enumerate(historial, start=1):
        print(f"{i}. {msg.type}: {msg.content}")
else:
    print("No se registró historial.")

=== 1. Configuración GitHub Models API ===
Base URL: https://models.inference.ai.azure.com
API Key configurada: ✓

=== 2. Configuración LangChain Model API ===
Modelo LangChain configurado: gpt-4o

=== 3. Configuración de memoria conversacional ===

=== 4. Streaming habilitado ===

=== CHATBOT INTERACTIVO BANCOESTADO ===
Escribe tu consulta y presiona Enter.
Para terminar, escribe: salir



Tú:  ¿Cual fue la primera pregunta que te hice?


Chatbot: Tu primera pregunta fue: **"¿Qué hago si pierdo mi tarjeta?"**. 😊

